In [1]:
import pandas as pd

df = pd.read_csv('data/fraud_scoring_dataset.csv')


In [2]:
df2 =  df.copy()
df2.head()

,transaction_amount,customer_age,account_age_days,num_transactions_30days,failed_login_attempts,international_transaction,fraud_score,first_name,last_name,country,country_risk_group
0,3751.655787,29.478242,955.964040,67.597596,5.719959,0,51.874785,Mia,Martinez,South Africa,medium_risk_country_group
1,9507.635921,51.597859,902.225638,79.871458,8.054323,1,5.116551,Henry,Jackson,Libya,high_risk_country_group
2,7322.619479,72.122642,3307.922964,25.796322,7.601609,0,23.285856,James,Harris,Somalia,high_risk_country_group
3,5990.598257,63.397943,911.594083,62.862536,1.538999,1,49.200173,Lucas,Thompson,Mexico,medium_risk_country_group
4,1568.626218,68.006791,993.344551,57.602852,1.492495,1,31.023528,Mia,Robinson,Iraq,high_risk_country_group


In [3]:
df_features = df2[["transaction_amount", "country", "customer_age", "country_risk_group"]].copy()
df_features.head()

,transaction_amount,country,customer_age,country_risk_group
0,3751.655787,South Africa,29.478242,medium_risk_country_group
1,9507.635921,Libya,51.597859,high_risk_country_group
2,7322.619479,Somalia,72.122642,high_risk_country_group
3,5990.598257,Mexico,63.397943,medium_risk_country_group
4,1568.626218,Iraq,68.006791,high_risk_country_group


In [4]:
# how many distinct countries in df_features.country
df_features.country.nunique()

30

In [5]:
# Z-Score Normalization: z = (x - μ) / σ
numeric_cols = ['transaction_amount', 'customer_age']

df_zscore = df_features.copy()

for col in numeric_cols:
    mean = df_zscore[col].mean()
    std = df_zscore[col].std()
    df_zscore[col] = (df_zscore[col] - mean) / std

print("Z-Score Normalized DataFrame:")
print(df_zscore.head(10))
print("\nVerification - Mean (should be ~0) and Std (should be ~1):")
print(df_zscore[numeric_cols].agg(['mean', 'std']))

Z-Score Normalized DataFrame:
   transaction_amount       country  customer_age         country_risk_group
0           -0.396103  South Africa     -1.101627  medium_risk_country_group
1            1.576169         Libya      0.119387    high_risk_country_group
2            0.827479       Somalia      1.252365    high_risk_country_group
3            0.371065        Mexico      0.770758  medium_risk_country_group
4           -1.144112          Iraq      1.025168    high_risk_country_group
5           -1.144195          Iraq      0.519409    high_risk_country_group
6           -1.479348        Turkey      0.634037  medium_risk_country_group
7            1.286791       Myanmar      1.171082    high_risk_country_group
8            0.379474          Iraq     -0.880760    high_risk_country_group
9            0.745595      Malaysia     -0.060209  medium_risk_country_group

Verification - Mean (should be ~0) and Std (should be ~1):
      transaction_amount  customer_age
mean       -4.440892e-17

In [6]:
# One-Hot Encoding for the 'country' column
df_encoded = pd.get_dummies(df_zscore, columns=['country'], drop_first=False)

print("One-Hot Encoded DataFrame:")
print(df_encoded.head())
print(f"\nShape: {df_encoded.shape}")
print(f"\nColumns: {df_encoded.columns.tolist()}")

One-Hot Encoded DataFrame:
   transaction_amount  customer_age         country_risk_group  \
0           -0.396103     -1.101627  medium_risk_country_group   
1            1.576169      0.119387    high_risk_country_group   
2            0.827479      1.252365    high_risk_country_group   
3            0.371065      0.770758  medium_risk_country_group   
4           -1.144112      1.025168    high_risk_country_group   

   country_Afghanistan  country_Brazil  country_Canada  country_Denmark  \
0                False           False           False            False   
1                False           False           False            False   
2                False           False           False            False   
3                False           False           False            False   
4                False           False           False            False   

   country_Finland  country_Germany  country_India  ...  country_Sudan  \
0            False            False          False 

In [7]:
df_features.head()

,transaction_amount,country,customer_age,country_risk_group
0,3751.655787,South Africa,29.478242,medium_risk_country_group
1,9507.635921,Libya,51.597859,high_risk_country_group
2,7322.619479,Somalia,72.122642,high_risk_country_group
3,5990.598257,Mexico,63.397943,medium_risk_country_group
4,1568.626218,Iraq,68.006791,high_risk_country_group


In [8]:
df_features.country_risk_group.value_counts(dropna=False)

country_risk_group
medium_risk_country_group    335
high_risk_country_group      335
safe_country_group           330
Name: count, dtype: int64

In [ ]:
# Typical classification problem